# EEG Preprocessing – OpenNeuro ds004584 (Parkinson's Disease)

This notebook performs **end-to-end EEG preprocessing** for the OpenNeuro ds004584 dataset.

## Dataset Information
- **Source**: OpenNeuro ds004584
- **Task**: Resting-state EEG
- **Subjects**: 150 (75 PD patients + 75 healthy controls)
- **Format**: EEGLAB (.set / .fdt)

## Preprocessing Pipeline
1. **Load raw EEG** - Read .set/.fdt files using MNE
2. **Channel selection** - Keep only EEG channels
3. **Filtering** - Bandpass (0.5-45 Hz) + Notch (50 Hz)
4. **Re-reference** - Average reference
5. **Artifact removal** - ICA for eye/muscle artifacts
6. **Resampling** - Downsample to 250 Hz
7. **Subject-wise split** - Train/Val/Test (70/15/15)
8. **Windowing** - Fixed-length epochs (2s, 50% overlap)
9. **Normalization** - Channel-wise z-score
10. **Save** - .npz format for model training

In [10]:
# =============================================================================
# Step 1: Import Required Libraries
# =============================================================================
# - os: File path operations
# - json: Save/load split information
# - numpy: Numerical operations and array handling
# - pandas: Read participants.tsv metadata
# - mne: EEG data processing (filtering, ICA, epoching)
# - sklearn: Train/val/test split with stratification

import os
import json
import numpy as np
import pandas as pd
import mne
from sklearn.model_selection import train_test_split

# Suppress verbose MNE output (show only warnings/errors)
mne.set_log_level("WARNING")

In [ ]:
# =============================================================================
# Step 2: Define Configuration Parameters
# =============================================================================

# --- Path Configuration ---
# RAW_ROOT: Path to raw OpenNeuro ds004584 dataset (relative to this notebook)
# OUT_ROOT: Path to save processed data
RAW_ROOT = "../data/raw/OpenNeuro_ds004584"
OUT_ROOT = "../data/processed/ds004584"
os.makedirs(OUT_ROOT, exist_ok=True)

# --- Filtering Parameters ---
LOW_FREQ = 0.5      # High-pass cutoff (Hz) - removes DC drift
HIGH_FREQ = 45      # Low-pass cutoff (Hz) - removes high-freq noise, keeps gamma
NOTCH_FREQ = 50     # Notch filter (Hz) - removes power line interference (50Hz in EU)
RESAMPLE_FREQ = 250 # Target sampling rate (Hz) - reduces computation

# --- Windowing Parameters ---
WINDOW_SEC = 2.0    # Window duration (seconds) - 2s = 500 samples at 250Hz
OVERLAP_SEC = 1.0   # Overlap between windows (seconds) - 50% overlap

# --- Reproducibility ---
RANDOM_STATE = 42   # Random seed for reproducible splits

In [12]:
# =============================================================================
# Step 3: Load Participant Metadata and Create Labels
# =============================================================================
# The participants.tsv file contains:
# - participant_id: Subject identifier (sub-001, sub-002, ...)
# - GROUP: Disease status ("PD" = Parkinson's, "Control" = Healthy)
# - AGE, GENDER, MOCA, UPDRS: Clinical information

# Read participant metadata
participants = pd.read_csv(
    os.path.join(RAW_ROOT, "participants.tsv"),
    sep="\t"
)

# Create binary labels: PD=1, Control=0
# Note: Column name is "GROUP" (uppercase) in this dataset
label_map = {"PD": 1, "Control": 0}
participants["label"] = participants["GROUP"].map(label_map)

# Display first few rows to verify
print(f"Total subjects: {len(participants)}")
print(f"PD patients: {(participants['label'] == 1).sum()}")
print(f"Controls: {(participants['label'] == 0).sum()}")
participants.head()

Total subjects: 149
PD patients: 100
Controls: 49


,participant_id,GROUP,ID,EEG,AGE,GENDER,MOCA,UPDRS,TYPE,label
0,sub-001,PD,1001,PD1001,80,M,19,28.0,1,1
1,sub-002,PD,1011,PD1011,81,M,17,25.0,1,1
2,sub-003,PD,1021,PD1021,68,F,26,10.0,1,1
3,sub-004,PD,1031,PD1031,80,M,22,10.0,1,1
4,sub-005,PD,1041,PD1041,56,M,21,13.0,1,1


In [13]:
# =============================================================================
# Step 4: Subject-wise Train/Validation/Test Split
# =============================================================================
# IMPORTANT: Split by SUBJECT, not by window/epoch
# This prevents data leakage (same subject in train and test)
#
# Split ratio: 70% train, 15% validation, 15% test
# Stratified split ensures balanced PD/Control ratio in each set

subjects = participants["participant_id"].values
labels = participants["label"].values

# First split: 70% train, 30% temp (val + test)
sub_train, sub_temp, y_train_labels, y_temp = train_test_split(
    subjects, labels,
    test_size=0.30,
    stratify=labels,
    random_state=RANDOM_STATE
)

# Second split: 50% of temp = 15% val, 15% test
sub_val, sub_test, y_val_labels, y_test_labels = train_test_split(
    sub_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

# Save split information for reproducibility
splits = {"train": sub_train, "val": sub_val, "test": sub_test}
with open(os.path.join(OUT_ROOT, "splits.json"), "w") as f:
    json.dump({k: list(v) for k, v in splits.items()}, f, indent=2)

print(f"Train subjects: {len(sub_train)} (PD: {sum(y_train_labels)}, Control: {len(y_train_labels)-sum(y_train_labels)})")
print(f"Val subjects:   {len(sub_val)} (PD: {sum(y_val_labels)}, Control: {len(y_val_labels)-sum(y_val_labels)})")
print(f"Test subjects:  {len(sub_test)} (PD: {sum(y_test_labels)}, Control: {len(y_test_labels)-sum(y_test_labels)})")

Train subjects: 104 (PD: 70, Control: 34)
Val subjects:   22 (PD: 15, Control: 7)
Test subjects:  23 (PD: 15, Control: 8)


In [14]:
# =============================================================================
# Step 5: Find Common EEG Channels Across All Subjects
# =============================================================================
# Different subjects may have different channel counts (e.g., 63 vs 64)
# We need to find channels that exist in ALL subjects to ensure consistent shape

def get_subject_channels(sub_id):
    """Get list of EEG channel names for a subject."""
    eeg_path = os.path.join(RAW_ROOT, sub_id, "eeg", f"{sub_id}_task-Rest_eeg.set")
    raw = mne.io.read_raw_eeglab(eeg_path, preload=False)
    raw.pick_types(eeg=True)
    return set(raw.ch_names)

print("Finding common channels across all subjects...")
all_subjects = participants["participant_id"].values

# Get channels from first subject
common_channels = get_subject_channels(all_subjects[0])

# Find intersection with all other subjects
for sub_id in all_subjects[1:]:
    sub_channels = get_subject_channels(sub_id)
    common_channels = common_channels.intersection(sub_channels)

# Convert to sorted list for consistency
COMMON_CHANNELS = sorted(list(common_channels))
print(f"Found {len(COMMON_CHANNELS)} common channels across {len(all_subjects)} subjects")
print(f"Channels: {COMMON_CHANNELS[:5]}... (showing first 5)")

Finding common channels across all subjects...


C:\Users\MSI\AppData\Local\Temp\ipykernel_21004\1518637754.py:10: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(eeg_path, preload=False)
C:\Users\MSI\AppData\Local\Temp\ipykernel_21004\1518637754.py:10: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(eeg_path, preload=False)
C:\Users\MSI\AppData\Local\Temp\ipykernel_21004\1518637754.py:10: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(eeg_path, preload=False)
C:\Users\MSI\AppData\Local\Temp\ipykernel_21004\1518637754.py:10: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around th

Found 60 common channels across 149 subjects
Channels: ['AF3', 'AF4', 'AF7', 'AF8', 'AFz']... (showing first 5)


C:\Users\MSI\AppData\Local\Temp\ipykernel_21004\1518637754.py:10: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(eeg_path, preload=False)


In [15]:
# =============================================================================
# Step 6: Define Preprocessing Function for Single Subject
# =============================================================================
# This function applies the full preprocessing pipeline to one subject's EEG:
# 1. Load raw EEG from EEGLAB .set file
# 2. Select only COMMON EEG channels (ensures consistent shape)
# 3. Bandpass filter (0.5-45 Hz) to keep relevant frequencies
# 4. Notch filter (50 Hz) to remove power line noise
# 5. Re-reference to average (common in EEG analysis)
# 6. ICA for artifact removal (eye blinks, muscle activity)
# 7. Resample to target frequency (reduces data size)

def preprocess_subject(sub_id):
    """
    Preprocess EEG data for a single subject.
    
    Parameters:
    -----------
    sub_id : str
        Subject identifier (e.g., "sub-001")
    
    Returns:
    --------
    raw : mne.io.Raw
        Preprocessed raw EEG object
    """
    # Construct path to EEG file
    eeg_path = os.path.join(
        RAW_ROOT, sub_id, "eeg", f"{sub_id}_task-Rest_eeg.set"
    )
    
    # Load raw EEG data
    raw = mne.io.read_raw_eeglab(eeg_path, preload=True)
    
    # Select only COMMON channels (ensures all subjects have same shape)
    raw.pick_channels(COMMON_CHANNELS, ordered=True)
    
    # Apply bandpass filter (0.5-45 Hz)
    # - High-pass (0.5 Hz): removes slow drifts
    # - Low-pass (45 Hz): removes high-freq noise, keeps up to gamma band
    raw.filter(LOW_FREQ, HIGH_FREQ)
    
    # Apply notch filter to remove power line interference
    raw.notch_filter(NOTCH_FREQ)
    
    # Re-reference to average of all EEG channels
    raw.set_eeg_reference("average")
    
    # ICA for artifact removal
    # - Use fewer components if we have fewer channels
    n_ica_components = min(20, len(COMMON_CHANNELS) - 1)
    ica = mne.preprocessing.ICA(
        n_components=n_ica_components,
        random_state=RANDOM_STATE,
        max_iter="auto"
    )
    ica.fit(raw)
    raw = ica.apply(raw)
    
    # Resample to reduce data size and computation
    raw.resample(RESAMPLE_FREQ)
    
    return raw

In [16]:
# =============================================================================
# Step 7: Define Function to Build Dataset for a Split
# =============================================================================
# This function processes all subjects in a split and creates windowed epochs:
# 1. For each subject: preprocess and create fixed-length epochs
# 2. Concatenate all epochs with corresponding labels and subject IDs
# 3. Returns arrays ready for model training

def build_split_dataset(subject_ids):
    """
    Build dataset for a set of subjects (train/val/test split).
    
    Parameters:
    -----------
    subject_ids : array-like
        List of subject identifiers to process
    
    Returns:
    --------
    X : np.ndarray
        EEG data array, shape (n_epochs, n_channels, n_samples)
    y : np.ndarray
        Labels array, shape (n_epochs,)
    sid : np.ndarray
        Subject IDs for each epoch (for analysis)
    """
    X_all, y_all, sid_all = [], [], []
    
    for i, sid in enumerate(subject_ids):
        print(f"  Processing {sid} ({i+1}/{len(subject_ids)})...")
        
        try:
            # Get label for this subject
            label = participants.loc[
                participants["participant_id"] == sid, "label"
            ].values[0]
            
            # Preprocess raw EEG
            raw = preprocess_subject(sid)
            
            # Create fixed-length epochs (windows)
            # - duration: window length in seconds
            # - overlap: overlap between consecutive windows
            epochs = mne.make_fixed_length_epochs(
                raw,
                duration=WINDOW_SEC,
                overlap=OVERLAP_SEC,
                preload=True
            )
            
            # Get epoch data as numpy array: (n_epochs, n_channels, n_samples)
            X = epochs.get_data()
            
            # Create label and subject ID arrays (same for all epochs from this subject)
            y = np.full(len(X), label)
            s = np.full(len(X), sid)
            
            X_all.append(X)
            y_all.append(y)
            sid_all.append(s)
            
            print(f"    -> {len(X)} epochs, shape: {X.shape[1:]} (channels x samples)")
            
        except Exception as e:
            print(f"    -> ERROR: {e}, skipping subject")
            continue
    
    return np.concatenate(X_all), np.concatenate(y_all), np.concatenate(sid_all)

In [17]:
# =============================================================================
# Step 8: Process All Splits (Train/Val/Test)
# =============================================================================
# This step takes the longest time as it processes all subjects
# Progress is printed for each subject

print("=" * 60)
print("Processing TRAINING set...")
print("=" * 60)
X_train, y_train, sid_train = build_split_dataset(sub_train)

print("\n" + "=" * 60)
print("Processing VALIDATION set...")
print("=" * 60)
X_val, y_val, sid_val = build_split_dataset(sub_val)

print("\n" + "=" * 60)
print("Processing TEST set...")
print("=" * 60)
X_test, y_test, sid_test = build_split_dataset(sub_test)

# Print summary
print("\n" + "=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Train: {X_train.shape} - PD: {(y_train==1).sum()}, Control: {(y_train==0).sum()}")
print(f"Val:   {X_val.shape} - PD: {(y_val==1).sum()}, Control: {(y_val==0).sum()}")
print(f"Test:  {X_test.shape} - PD: {(y_test==1).sum()}, Control: {(y_test==0).sum()}")

C:\Users\MSI\AppData\Local\Temp\ipykernel_21004\1875097408.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(eeg_path, preload=True)


Processing TRAINING set...
  Processing sub-032 (1/104)...
    -> 150 epochs, shape: (60, 500) (channels x samples)
  Processing sub-126 (2/104)...


C:\Users\MSI\AppData\Local\Temp\ipykernel_21004\1875097408.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(eeg_path, preload=True)


    -> 165 epochs, shape: (60, 500) (channels x samples)
  Processing sub-076 (3/104)...


C:\Users\MSI\AppData\Local\Temp\ipykernel_21004\1875097408.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(eeg_path, preload=True)


...

    -> 124 epochs, shape: (60, 500) (channels x samples)
  Processing sub-097 (22/23)...


C:\Users\MSI\AppData\Local\Temp\ipykernel_21004\1875097408.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(eeg_path, preload=True)


    -> 158 epochs, shape: (60, 500) (channels x samples)
  Processing sub-121 (23/23)...


C:\Users\MSI\AppData\Local\Temp\ipykernel_21004\1875097408.py:33: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(eeg_path, preload=True)
c:\Users\MSI\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\decomposition\_fastica.py:127: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


    -> 149 epochs, shape: (60, 500) (channels x samples)

DATASET SUMMARY
Train: (16255, 60, 500) - PD: 10576, Control: 5679
Val:   (3668, 60, 500) - PD: 2429, Control: 1239
Test:  (3767, 60, 500) - PD: 2432, Control: 1335


In [19]:
# =============================================================================
# Step 9: Normalize Data (Z-score Normalization)
# =============================================================================
# Apply channel-wise z-score normalization:
# - Compute mean and std from TRAINING data only (prevent data leakage)
# - Apply same normalization to val and test sets
#
# Formula: X_norm = (X - mean) / std
# This centers data around 0 with unit variance, helping model convergence

# Compute statistics from training data only
# axis=(0, 2): compute mean/std across epochs and time samples, keeping channels
mean = X_train.mean(axis=(0, 2), keepdims=True)
std = X_train.std(axis=(0, 2), keepdims=True) + 1e-8  # Small epsilon to prevent division by zero

# Apply normalization to all splits
X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

print("Normalization complete!")
print(f"Training data - Mean: {X_train.mean():.6f}, Std: {X_train.std():.6f}")

Normalization complete!
Training data - Mean: -0.000000, Std: 0.999336


In [20]:
# =============================================================================
# Step 10: Save Processed Data
# =============================================================================
# Save preprocessed data in NumPy compressed format (.npz)
# Each file contains:
# - X: EEG data (n_epochs, n_channels, n_samples)
# - y: Labels (n_epochs,) - 0=Control, 1=PD
# - subject_id: Subject IDs for each epoch (for subject-level analysis)

np.savez(
    os.path.join(OUT_ROOT, "train.npz"),
    X=X_train, y=y_train, subject_id=sid_train
)
np.savez(
    os.path.join(OUT_ROOT, "val.npz"),
    X=X_val, y=y_val, subject_id=sid_val
)
np.savez(
    os.path.join(OUT_ROOT, "test.npz"),
    X=X_test, y=y_test, subject_id=sid_test
)

# Also save normalization parameters for inference
np.savez(
    os.path.join(OUT_ROOT, "norm_params.npz"),
    mean=mean, std=std
)

# Save channel names for reference
with open(os.path.join(OUT_ROOT, "channels.json"), "w") as f:
    json.dump(COMMON_CHANNELS, f, indent=2)

print("=" * 60)
print("PREPROCESSING COMPLETE!")
print("=" * 60)
print(f"Files saved to: {OUT_ROOT}")
print(f"  - train.npz: {X_train.shape}")
print(f"  - val.npz:   {X_val.shape}")
print(f"  - test.npz:  {X_test.shape}")
print(f"  - norm_params.npz: mean & std for inference")
print(f"  - channels.json: {len(COMMON_CHANNELS)} channel names")
print(f"  - splits.json: subject IDs per split")

PREPROCESSING COMPLETE!
Files saved to: ../data/processed/ds004584
  - train.npz: (16255, 60, 500)
  - val.npz:   (3668, 60, 500)
  - test.npz:  (3767, 60, 500)
  - norm_params.npz: mean & std for inference
  - channels.json: 60 channel names
  - splits.json: subject IDs per split
